In [31]:
from atomworks.io.utils.visualize import view

In [49]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/Users/wzy/Documents/论文复现/"
    "foundry-production/results/rfd3_enzyme"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("输出目录：", OUTPUT_DIR)

输出目录： /Users/wzy/Documents/论文复现/foundry-production/results/rfd3_enzyme


In [32]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from rfd3.inference.input_parsing import DesignInputSpecification
conf = RFD3InferenceConfig(
    ckpt_path='/Users/wzy/Documents/论文复现/foundry-production/checkpoints/rfd3_latest.ckpt',
    diffusion_batch_size=8,
    dump_trajectories=True,
    inference_sampler={"num_timesteps": 200},
)
# Lazily initialize model
model = RFD3InferenceEngine(**conf)

In [33]:
spec = DesignInputSpecification(
    input="/Users/wzy/Documents/论文复现/foundry-production/models/rfd3/docs/tutorials/enzyme_tutorial_files/1euv_lig.pdb",

    ligand="l:g",

    unindex="A514,A531,A574,A579-581",

    length="100-200",

    ori_token=[0, 1, 0],

    select_fixed_atoms={
        "A514": "NE2,CE1,ND1,CD2,CG,CB",
        "A531": "OD1,CG,OD2,CB",
        "A574": "NE2,CD,OE1,CG",
        "A579": "C,O,CA,N",
        "A580": "SG,CB,CA,N,C,O",
        "A581": "C,O,CA,N",
    },

    select_buried={
        "l:g": "O1,C8,O3,C4,C5,C23,C24,C25,C26,C27"
    },

    select_exposed={
        "l:g": "C2,C22,C19,C18,C17,C20,C16,C15,O21,O14,C13,C12"
    },

    select_unfixed_sequence="A579,A581",

)

view(spec.build())


16:26:27 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [34]:
outputs = model.run(
    inputs=spec,
    n_batches=1,
)

16:26:29 INFO foundry.utils.ddp: Apple MPS detected - using MPS accelerator with float32 precision
16:26:33 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
16:26:33 WARNING atomworks.ml: Cached data not found for ALA at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/A/ALA/ALA.pt
16:26:33 WARNING atomworks.ml: Cached data not found for ASP at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/A/ASP/ASP.pt
16:26:33 WARNING atomworks.ml: Cached data not found for CYS at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/C/CYS/CYS.pt
16:26:33 WARNING atomworks.ml: Cached data not found for GLN at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/G/GLN/GLN.pt
16:26:33 WARNING atomworks.ml: Cached data not found for GLY at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/G/GLY/GLY.pt
16:26:33 WARNING atomworks.ml: Cached data not found for HIS at /net/tukwila/ncorley/datahub/MACE-OFF23_medium/H/HIS/HIS.pt
16:26:33 WARNING atomworks.ml: Cached data not foun

In [35]:
for idx, data in outputs.items():
    print(f"Output type for batch {idx}: {type(data)}[0] = {type(data[0])}")
    print(f"Output atom_array: {data[0].atom_array}")
    atom_array = data[0].atom_array
view(atom_array)

Output type for batch backbone_0_0: <class 'list'>[0] = <class 'rfd3.engine.RFD3Output'>
Output atom_array:     A       1  ALA N      N        17.626    8.782    4.028
    A       1  ALA CA     C        16.445    8.790    3.206
    A       1  ALA C      C        16.674    7.961    1.950
    A       1  ALA O      O        17.553    8.244    1.150
    A       1  ALA CB     C        16.071   10.222    2.827
    A       2  VAL N      N        15.870    6.932    1.783
    A       2  VAL CA     C        15.963    6.063    0.621
    A       2  VAL C      C        15.364    6.743   -0.597
    A       2  VAL O      O        15.929    6.651   -1.690
    A       2  VAL CB     C        15.271    4.724    0.900
    A       2  VAL CG1    C        15.252    3.859   -0.340
    A       2  VAL CG2    C        15.954    3.998    2.044
    A       3  LEU N      N        14.265    7.442   -0.438
    A       3  LEU CA     C        13.592    8.190   -1.497
    A       3  LEU C      C        13.677    9.673  

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [43]:
first_key = next(iter(outputs))
rfd3_results = outputs[first_key]

print("生成结构总数：", len(rfd3_results))

assert len(rfd3_results) == 8

生成结构总数： 8


In [140]:
import pandas as pd
candidate_rows = []

for idx, result in enumerate(rfd3_results):
    metrics = result.metadata.get("metrics", {})
    candidate_rows.append({
        "candidate_idx": idx,
        "join_point_rmsd": metrics.get("join_point_rmsd"),
        "max_ca_deviation": metrics.get("max_ca_deviation"),
        "n_chainbreaks": metrics.get("n_chainbreaks"),
        "ligand_clashes": metrics.get("n_clashing.ligand_clashes"),
        "ligand_min_distance": metrics.get("n_clashing.ligand_min_distance"),
        "helix_fraction": metrics.get("helix_fraction"),
        "sheet_fraction": metrics.get("sheet_fraction"),
        "loop_fraction": metrics.get("loop_fraction"),
        "num_residues": metrics.get("num_residues"),
    })

candidate_table = pd.DataFrame(candidate_rows).set_index("candidate_idx")
candidate_table

,join_point_rmsd,max_ca_deviation,n_chainbreaks,ligand_clashes,ligand_min_distance,helix_fraction,sheet_fraction,loop_fraction,num_residues
candidate_idx,,,,,,,,,
0,0.485252,0.055959,0,0,3.019134,0.140741,0.400000,0.459259,138
1,0.231622,0.100192,0,0,1.917231,0.733333,0.000000,0.266667,138
2,0.238365,0.264769,0,0,3.696963,0.311111,0.148148,0.540741,138
3,0.384576,0.055959,0,0,3.137229,0.755556,0.000000,0.244444,138
4,0.470296,0.124219,0,0,3.633881,0.592593,0.133333,0.274074,138
5,0.341777,0.055959,0,0,2.732502,0.688889,0.000000,0.311111,138
6,0.629074,1.190680,2,0,3.207911,0.533333,0.140741,0.325926,138
7,0.321593,0.124444,0,0,2.874780,0.792593,0.000000,0.207407,138


In [160]:
# ==================== 唯一需要修改的 RFD3 候选索引 ====================
candidate_idx = 4
# =====================================================================

assert 0 <= candidate_idx < len(rfd3_results)

rfd3_result = rfd3_results[candidate_idx]
rfd3_atom_array = rfd3_result.atom_array
index_map = {
    str(original): str(generated)
    for original, generated in rfd3_result.metadata["diffused_index_map"].items()
}

print("当前 Candidate:", candidate_idx)
print("原始残基 → 新残基编号:")
for original, generated in index_map.items():
    print(f"  {original} -> {generated}")

display(candidate_table.loc[[candidate_idx]])
view(rfd3_atom_array)

当前 Candidate: 4
原始残基 → 新残基编号:
  A514 -> A122
  A531 -> A88
  A574 -> A82
  A579 -> A63
  A580 -> A64
  A581 -> A65


,join_point_rmsd,max_ca_deviation,n_chainbreaks,ligand_clashes,ligand_min_distance,helix_fraction,sheet_fraction,loop_fraction,num_residues
candidate_idx,,,,,,,,,
4,0.470296,0.124219,0,0,3.633881,0.592593,0.133333,0.274074,138


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [161]:
fixed_original = ["A514", "A531", "A574", "A580"]
fixed_generated = [index_map[residue] for residue in fixed_original]

print("LigandMPNN 固定残基:", fixed_generated)

LigandMPNN 固定残基: ['A122', 'A88', 'A82', 'A64']


In [162]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

mpnn_engine = MPNNInferenceEngine(
    checkpoint_path="/Users/wzy/Documents/论文复现/foundry-production/checkpoints/ligandmpnn_v_32_010_25.pt",
    model_type="ligand_mpnn",
    is_legacy_weights=True,
    out_directory=None,
    write_structures=False,
    write_fasta=False,
)

mpnn_input_configs = [{
    "name": f"candidate_{candidate_idx}",
    "batch_size": 10,
    "number_of_batches": 1,
    "temperature": 0.1,
    "remove_waters": True,
    "fixed_residues": fixed_generated,
    "atomize_side_chains": True,
}]

mpnn_outputs = mpnn_engine.run(
    input_dicts=mpnn_input_configs,
    atom_arrays=[rfd3_atom_array],  
)

print("LigandMPNN 序列数量:", len(mpnn_outputs))
assert len(mpnn_outputs) == 10

16:14:25 WARNING atomworks.io.utils.ccd: The following CCD codes were not found in Biotite's built-in CCD: {np.str_('l:g')}


LigandMPNN 序列数量: 10


In [163]:
from biotite.sequence import ProteinSequence
from biotite.structure import get_residue_starts, rmsd, superimpose
def sequence_from_atom_array(atom_array):
    protein_mask = ~atom_array.hetero
    protein = atom_array[protein_mask]
    starts = get_residue_starts(protein)
    return "".join(
        ProteinSequence.convert_letter_3to1(str(res_name))
        for res_name in protein.res_name[starts]
    )


sequence_rows = []
MPNN_OUTPUT_DIR = OUTPUT_DIR / "mpnn"

for sequence_idx, item in enumerate(mpnn_outputs):
    sequence_atom_array = sequence_from_atom_array(item.atom_array)
    sequence_reported = item.output_dict.get("designed_sequence", sequence_atom_array)
    sequence_rows.append({
        "candidate_idx": candidate_idx,
        "sequence_idx": sequence_idx,
        "sequence": sequence_reported,
        "length": len(sequence_reported),
    })
    to_cif_file(
        item.atom_array,
        MPNN_OUTPUT_DIR / f"candidate_{candidate_idx}_sequence_{sequence_idx}_target.cif",
    )

sequences_df = pd.DataFrame(sequence_rows)
sequences_df

,candidate_idx,sequence_idx,sequence,length
0,4,0,TVSVTLTDVTLDRAIALARQTPGGAAVAAQLEALRARFEAAAPRSL...,138
1,4,1,MVSVTLTDITLDRAIALARQTPGGAALAAQLEALRAAFEAAAPQSR...,138
2,4,2,TVSVTLTNITLDAAIALARATPGGAALAAQLEALRAAFEAAAPRSL...,138
3,4,3,TVSVTLENITLDRAIALARQTPGGAALAAQLEALRAAFEAAAPRSL...,138
4,4,4,MVSVTLEDVTLDRAIALARQTPGGAALAAQLEGLRALFEAAAPQSL...,138
5,4,5,MVSVTLTDVTLDRAIALAEATPGGAAVAAMLRELRARFEAAAPQSL...,138
6,4,6,TVSVTLTDITLDRAIALARATPGGAALAAQLEALRARFEAAAPRSL...,138
7,4,7,TVSVTLTDVTLDRAIALARETPGGAAVAAQLEALRAAFEAAAPRSL...,138
8,4,8,TVSVTLTDITLDKAIALARATPGGAAVAAQLEALRAAFEAAAPRSL...,138
9,4,9,TVSVTLTDITLDRAIALAEATPGGAALAAQLRALRARFEAAAPQSL...,138


In [164]:
RUN_ALL_SEQUENCES = True
TEST_SEQUENCE_IDX = 0

if RUN_ALL_SEQUENCES:
    rf3_sequence_indices = list(range(len(mpnn_outputs)))
else:
    rf3_sequence_indices = [TEST_SEQUENCE_IDX]

print("本次 RF3 序列索引:", rf3_sequence_indices)
print("预计结构样本数:", len(rf3_sequence_indices) * 5)

本次 RF3 序列索引: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
预计结构样本数: 50


In [165]:
rf3_inputs = []
example_to_sequence_idx = {}

for sequence_idx in rf3_sequence_indices:
    example_id = f"candidate_{candidate_idx}_sequence_{sequence_idx}"
    selected_mpnn_atom_array = mpnn_outputs[sequence_idx].atom_array
    rf3_inputs.append(
        InferenceInput.from_atom_array(
            selected_mpnn_atom_array,
            example_id=example_id,
        )
    )
    example_to_sequence_idx[example_id] = sequence_idx

rf3_all_outputs = rf3_engine.run(
    inputs=rf3_inputs,
    out_dir=None,
)

print("RF3 example 数量:", len(rf3_all_outputs))
for example_id, samples in rf3_all_outputs.items():
    print(example_id, "->", len(samples), "samples")

16:14:40 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
16:14:40 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
16:14:40 WARNING atomworks.io.utils.ccd: The following CCD codes were not found in Biotite's built-in CCD: {np.str_('l:g')}
16:14:40 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
16:14:40 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
16:14:40 WARNING atomworks.io.

RF3 example 数量: 10
candidate_4_sequence_0 -> 5 samples
candidate_4_sequence_1 -> 5 samples
candidate_4_sequence_2 -> 5 samples
candidate_4_sequence_3 -> 5 samples
candidate_4_sequence_4 -> 5 samples
candidate_4_sequence_5 -> 5 samples
candidate_4_sequence_6 -> 5 samples
candidate_4_sequence_7 -> 5 samples
candidate_4_sequence_8 -> 5 samples
candidate_4_sequence_9 -> 5 samples


In [166]:
BACKBONE_ATOMS = {"N", "CA", "C", "O"}


def keyed_backbone_coords(atom_array):
    coords = {}
    for chain_id, res_id, atom_name, hetero, coord in zip(
        atom_array.chain_id,
        atom_array.res_id,
        atom_array.atom_name,
        atom_array.hetero,
        atom_array.coord,
    ):
        atom_name = str(atom_name)
        if bool(hetero) or atom_name not in BACKBONE_ATOMS:
            continue
        key = (str(chain_id), int(res_id), atom_name)
        coords[key] = np.asarray(coord, dtype=np.float32)
    return coords


def matched_backbone_rmsd(reference, predicted):
    reference_coords = keyed_backbone_coords(reference)
    predicted_coords = keyed_backbone_coords(predicted)
    common_keys = sorted(set(reference_coords) & set(predicted_coords))
    if len(common_keys) < 3:
        raise ValueError("共同主链原子不足，无法计算 RMSD")

    ref = np.stack([reference_coords[key] for key in common_keys])
    pred = np.stack([predicted_coords[key] for key in common_keys])
    pred_fitted, _ = superimpose(ref, pred)
    return float(rmsd(ref, pred_fitted)), len(common_keys)


result_rows = []

for example_id, samples in rf3_all_outputs.items():
    sequence_idx = example_to_sequence_idx[example_id]
    target = mpnn_outputs[sequence_idx].atom_array

    for sample in samples:
        rmsd_value, n_matched_atoms = matched_backbone_rmsd(
            target,
            sample.atom_array,
        )
        summary = sample.summary_confidences
        result_rows.append({
            "candidate_idx": candidate_idx,
            "sequence_idx": sequence_idx,
            "sample_idx": sample.sample_idx,
            "seed": sample.seed,
            "backbone_rmsd": rmsd_value,
            "matched_backbone_atoms": n_matched_atoms,
            "ranking_score": summary.get("ranking_score"),
            "overall_plddt": summary.get("overall_plddt"),
            "overall_pae": summary.get("overall_pae"),
            "ptm": summary.get("ptm"),
            "iptm": summary.get("iptm"),
            "has_clash": summary.get("has_clash"),
        })

results_df = pd.DataFrame(result_rows).sort_values(
    ["sequence_idx", "sample_idx"]
).reset_index(drop=True)

results_df.to_csv(OUTPUT_DIR / "rf3_all_results.csv", index=False)
results_df

,candidate_idx,sequence_idx,sample_idx,seed,backbone_rmsd,matched_backbone_atoms,ranking_score,overall_plddt,overall_pae,ptm,iptm,has_clash
0,4,0,0,0,2.012366,552,0.5753,0.8486,6.8578,0.833017,0.510813,False
1,4,0,1,0,1.953441,552,0.5731,0.8480,6.8161,0.832446,0.508296,False
2,4,0,2,0,1.845808,552,0.5711,0.8481,6.8200,0.831730,0.505900,False
3,4,0,3,0,1.837984,552,0.5787,0.8484,6.7928,0.834111,0.514785,False
4,4,0,4,0,1.889724,552,0.5688,0.8486,6.8660,0.832084,0.503027,False
5,4,1,0,0,1.880157,552,0.5830,0.8502,6.4216,0.837184,0.519448,False
6,4,1,1,0,2.076102,552,0.5744,0.8510,6.4782,0.836084,0.508959,False
7,4,1,2,0,1.887430,552,0.5886,0.8502,6.4066,0.837572,0.526350,False
8,4,1,3,0,2.132432,552,0.5838,0.8506,6.4338,0.837154,0.520442,False
9,4,1,4,0,2.000413,552,0.5875,0.8506,6.4351,0.837094,0.525069,False


In [167]:
best_rmsd_by_sequence = (
    results_df.loc[
        results_df.groupby("sequence_idx")["backbone_rmsd"].idxmin()
    ]
    .sort_values("sequence_idx")
    .reset_index(drop=True)
)

best_ranked_by_sequence = (
    results_df.loc[
        results_df.groupby("sequence_idx")["ranking_score"].idxmax()
    ]
    .sort_values("sequence_idx")
    .reset_index(drop=True)
)

print("每条序列 RMSD 最低的 sample：")
display(best_rmsd_by_sequence)

print("每条序列 ranking_score 最高的 sample：")
display(best_ranked_by_sequence)

每条序列 RMSD 最低的 sample：


,candidate_idx,sequence_idx,sample_idx,seed,backbone_rmsd,matched_backbone_atoms,ranking_score,overall_plddt,overall_pae,ptm,iptm,has_clash
0,4,0,3,0,1.837984,552,0.5787,0.8484,6.7928,0.834111,0.514785,False
1,4,1,0,0,1.880157,552,0.5830,0.8502,6.4216,0.837184,0.519448,False
2,4,2,0,0,1.211354,552,0.5774,0.8490,6.7913,0.835456,0.512851,False
3,4,3,1,0,2.289334,552,0.5719,0.8384,6.7868,0.828503,0.507705,False
4,4,4,1,0,1.430791,552,0.6110,0.8516,6.1149,0.854368,0.550180,False
5,4,5,1,0,0.817255,552,0.5904,0.8462,6.3460,0.842956,0.527229,False
6,4,6,3,0,1.496940,552,0.6039,0.8475,6.6129,0.839547,0.544986,False
7,4,7,0,0,1.844428,552,0.5952,0.8693,5.9124,0.863090,0.528239,False
8,4,8,2,0,1.024156,552,0.5678,0.8497,6.9025,0.823379,0.503939,False
9,4,9,0,0,1.624609,552,0.5925,0.8582,6.4140,0.846363,0.528976,False


每条序列 ranking_score 最高的 sample：


,candidate_idx,sequence_idx,sample_idx,seed,backbone_rmsd,matched_backbone_atoms,ranking_score,overall_plddt,overall_pae,ptm,iptm,has_clash
0,4,0,3,0,1.837984,552,0.5787,0.8484,6.7928,0.834111,0.514785,False
1,4,1,2,0,1.887430,552,0.5886,0.8502,6.4066,0.837572,0.526350,False
2,4,2,1,0,1.418587,552,0.5832,0.8486,6.7716,0.834510,0.520414,False
3,4,3,3,0,2.557383,552,0.5737,0.8388,6.7846,0.829782,0.509682,False
4,4,4,2,0,1.446008,552,0.6116,0.8515,6.1043,0.854111,0.550983,False
5,4,5,1,0,0.817255,552,0.5904,0.8462,6.3460,0.842956,0.527229,False
6,4,6,1,0,1.660781,552,0.6048,0.8474,6.6103,0.839907,0.546033,False
7,4,7,1,0,1.913939,552,0.5959,0.8690,5.9163,0.863156,0.529116,False
8,4,8,2,0,1.024156,552,0.5678,0.8497,6.9025,0.823379,0.503939,False
9,4,9,2,0,1.798659,552,0.5935,0.8580,6.3910,0.845628,0.530463,False


In [168]:
# ==================== 任意选择一个 RF3 结果 ====================
EXPORT_SEQUENCE_IDX = 5
EXPORT_SAMPLE_IDX = 1
# ==============================================================

export_example_id = f"candidate_{candidate_idx}_sequence_{EXPORT_SEQUENCE_IDX}"
assert export_example_id in rf3_all_outputs, f"没有运行：{export_example_id}"

matching_samples = [
    sample
    for sample in rf3_all_outputs[export_example_id]
    if sample.sample_idx == EXPORT_SAMPLE_IDX
]
assert len(matching_samples) == 1

selected_mpnn_output = mpnn_outputs[EXPORT_SEQUENCE_IDX]
selected_rf3_output = matching_samples[0]


def pymol_safe_copy(atom_array):
    # 把自定义配体名 l:g 改为 LIG，避免 PyMOL 尝试联网下载 l:g.cif。
    safe = atom_array.copy()
    ligand_mask = safe.hetero & (safe.res_name == "l:g")
    safe.res_name[ligand_mask] = "LIG"
    return safe


PYMOL_DIR = OUTPUT_DIR / "pymol"
generated_path = PYMOL_DIR / (
    f"candidate_{candidate_idx}_sequence_{EXPORT_SEQUENCE_IDX}_target.cif"
)
refolded_path = PYMOL_DIR / (
    f"candidate_{candidate_idx}_sequence_{EXPORT_SEQUENCE_IDX}_sample_{EXPORT_SAMPLE_IDX}_rf3.cif"
)

to_cif_file(pymol_safe_copy(selected_mpnn_output.atom_array), generated_path)
to_cif_file(pymol_safe_copy(selected_rf3_output.atom_array), refolded_path)

print("目标骨架:", generated_path)
print("RF3 重折叠:", refolded_path)

目标骨架: /Users/wzy/Documents/论文复现/foundry-production/results/rfd3_enzyme/pymol/candidate_4_sequence_5_target.cif
RF3 重折叠: /Users/wzy/Documents/论文复现/foundry-production/results/rfd3_enzyme/pymol/candidate_4_sequence_5_sample_1_rf3.cif
